# RNN Models Demo
This notebook loads the trained states of the Eminem Lyric Generator and the Tomita 2 Classifier to demonstrate their capabilities.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Eminem Lyric Generator
This model generates lyrics based on a starting string.

In [ ]:
class LyricRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(LyricRNN, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, h):
        x = self.embed(x)
        out, h = self.gru(x, h)
        out = self.fc(out.reshape(out.size(0) * out.size(1), out.size(2)))
        return out, h

def generate(model, start_str, char_to_int, int_to_char, length=200, temperature=0.7):
    model.eval()
    chars = [ch for ch in start_str]
    input_seq = torch.tensor([[char_to_int[ch] for ch in start_str]]).to(device)
    h = None
    
    with torch.no_grad():
        for _ in range(length):
            output, h = model(input_seq, h)
            output_dist = (output[-1] / max(temperature, 1e-6)).exp()
            top_ch_tensor = torch.multinomial(output_dist, 1)[0]
            top_ch = int(top_ch_tensor.item())
            chars.append(int_to_char[top_ch])
            input_seq = torch.tensor([[top_ch]]).to(device)
            
    return ''.join(chars)

# Load Model
eminem_checkpoint = torch.load('models/eminem_rnn.pth', map_location=device)
eminem_model = LyricRNN(
    eminem_checkpoint['vocab_size'], 
    eminem_checkpoint['embed_size'], 
    eminem_checkpoint['hidden_size'], 
    eminem_checkpoint['num_layers']
).to(device)
eminem_model.load_state_dict(eminem_checkpoint['model_state_dict'])

char_to_int = eminem_checkpoint['char_to_int']
int_to_char = eminem_checkpoint['int_to_char']

print("Eminem Lyric Generator Loaded.")

In [ ]:
# Example Generation
seed_text = "Moms spaghetti"
generated_lyrics = generate(eminem_model, seed_text, char_to_int, int_to_char, length=150, temperature=0.8)
print(f"Seed: {seed_text}\n---\n{generated_lyrics}")

## 2. Tomita 2 Classifier
This model classifies whether a string belongs to Tomita Grammar 2: `(0*11)*0*`.

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, hidden_size=10):
        super().__init__()
        self.rnn = nn.RNN(2, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # Simple forward without lengths for demo purposes
        _, h = self.rnn(x)
        return self.sigmoid(self.fc(h.squeeze(0)))

def to_tensor(s):
    t = torch.zeros(1, len(s) if s else 1, 2)
    for i, char in enumerate(s): t[0, i, int(char)] = 1.0
    return t

# Load Model
tomita_checkpoint = torch.load('models/tomita2_rnn.pth', map_location=device)
tomita_model = SimpleRNN(hidden_size=tomita_checkpoint['hidden_size']).to(device)
tomita_model.load_state_dict(tomita_checkpoint['model_state_dict'])
tomita_model.eval()

print("Tomita 2 Classifier Loaded.")

In [ ]:
def test_tomita(s):
    with torch.no_grad():
        pred = tomita_model(to_tensor(s).to(device))
        result = pred.item() > 0.5
        print(f"String: '{s}' | Belongs to Tomita 2: {result} (Score: {pred.item():.4f})")

test_strings = ["11", "00110", "1", "01", "1111", "000"]
for s in test_strings:
    test_tomita(s)